In [2]:
import pandas as pd
import numpy as np

data = np.load("train.npz")
print(data.files)

for name in data.files:
    arr = data[name]
    print(name,arr.shape ,arr.dtype)

print (data["input_pos"][60][1])

['input_pos', 'future_pos', 'state_gt', 'regime', 'seg_id', 'dt']
input_pos (22888, 50, 3) float32
future_pos (22888, 40, 3) float32
state_gt (22888, 50, 6) float32
regime (22888,) int64
seg_id (22888,) int64
dt () float64
[ -52.331013   18.221224 -124.57968 ]


In [27]:
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense, Reshape, Input
model = Sequential()
model.add(Input(shape=(50,3)))
model.add(LSTM(units = 64))
model.add(Dense(120))
model.add(Reshape((40,3)))
model.compile(optimizer = 'adam', loss = 'mse')

In [28]:
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 120)            │         7,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 40, 3)          │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,208 (98.47 KB)

 Trainable params: 25,208 (98.47 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
data_val = np.load("val.npz")

In [30]:
print(data_val.files)

['input_pos', 'future_pos', 'state_gt', 'regime', 'seg_id', 'dt']


In [31]:
#chaninging the func so the sigma in the training data is varied

In [32]:
def prep_varied(d, reps, rng, lo=0.05, hi=1.0):
    Xs, ys = [], []
    for _ in range(reps):
        s = rng.uniform(lo, hi, (len(d["input_pos"]), 1, 1))
        X = d["input_pos"] + rng.normal(0, 1, d["input_pos"].shape) * s
        anchor = X[:, -1:, :]
        Xs.append((X - anchor).astype(np.float32))
        ys.append((d["future_pos"] - anchor).astype(np.float32))
    return np.concatenate(Xs), np.concatenate(ys)

def prep_fixed(d, sigma, rng):
    X = d["input_pos"] + rng.normal(0, sigma, d["input_pos"].shape).astype(np.float32)
    anchor = X[:, -1:, :]
    return (X - anchor).astype(np.float32), (d["future_pos"] - anchor).astype(np.float32)

rng = np.random.default_rng(0)
Xtr_noisy, ytr = prep_varied(data, reps=3, rng=rng)
Xva_noisy, yva = prep_fixed(data_val, sigma=0.3, rng=rng)

print(Xtr_noisy.shape, ytr.shape)

(68664, 50, 3) (68664, 40, 3)


In [24]:
print(Xtr_noisy.shape, ytr.shape, Xva_noisy.shape, yva.shape)

(22888, 50, 3) (22888, 40, 3) (4302, 50, 3) (4302, 40, 3)


In [34]:
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

cbs = [
    EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
    ModelCheckpoint(filepath="128_sigma03.keras", monitor="val_loss", save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5)
]

hist = model.fit(Xtr_noisy, ytr,
                validation_data = (Xva_noisy, yva),
                 epochs = 200,
                 batch_size = 64,
                 callbacks = cbs
                )


Epoch 1/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - loss: 21.8004 - val_loss: 8.8705 - learning_rate: 0.0010
Epoch 2/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - loss: 12.4836 - val_loss: 8.1839 - learning_rate: 0.0010
Epoch 3/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - loss: 11.5088 - val_loss: 7.7869 - learning_rate: 0.0010
Epoch 4/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - loss: 10.8195 - val_loss: 7.5307 - learning_rate: 0.0010
Epoch 5/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - loss: 10.2025 - val_loss: 7.3233 - learning_rate: 0.0010
Epoch 6/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 9.6819 - val_loss: 7.1746 - learning_rate: 0.0010
Epoch 7/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - loss: 9.2379 - val_loss: 6.9631 - learning_rate: 0.0010
Epoch 8/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - loss: 8.8937 - val_loss: 6.7266 - learning_rate: 0.0010
Epoch 9/200
1073/1073 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - loss: 8.6401 - v

In [36]:
#best val_loss for 64 units 6.20 with 115 on epochs
#best val_loss for 128 units 6.15 with 39 epochs but but train_loss 4.14 was below val_loss
#-> 128 units is overfitting because both sigmas are 0.3(the same)
#but I still need to have a loop for sigma for eliza to do the interpretation
#sigmas are diff so the diff btw tr_loss and val_loss has no meaning 

import os
print(os.getcwd())

In [18]:
import os
os.rename("best_lstm.keras", "lstm64_sigma03.keras")
print(os.listdir("."))

['train.npz', 'lstm64_sigma03.keras', 'positions_20260728_123829.csv', 'Trajectory model.ipynb', 'val.npz', 'LSTM_on_npz.ipynb', '.ipynb_checkpoints']


In [37]:
import os
os.rename("128_sigma03.keras", "varsigma_64.keras")
print(os.listdir("."))

['varsigma_64.keras', 'train.npz', 'lstm64_sigma03.keras', 'positions_20260728_123829.csv', 'Trajectory model.ipynb', 'val.npz', 'LSTM_on_npz.ipynb', '.ipynb_checkpoints']
